In [1]:
import cv2  # Biblioteca para procesamiento de imágenes y video
import mediapipe as mp  # Framework de Google para visión por computadora
import numpy as np  # Biblioteca para cálculos numéricos
import time  # Para gestionar el tiempo y calcular duraciones
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
from collections import deque

# ==============================
# CONFIGURACIÓN DE PLOTEO
# ==============================
plt.style.use("dark_background")
plt.ioff()
fig, ax = plt.subplots(figsize=(6, 3), dpi=100)
canvas = FigureCanvas(fig)

ear_values = []
frame_numbers = []
max_frames = 100
EAR_THRESHOLD = None  # Se calibrará al inicio

x_vals = list(range(max_frames))
y_vals = [0] * max_frames
Y_vals = [0] * max_frames

EAR_curve, = ax.plot(x_vals, y_vals, color="#56f10d", label="EAR", linewidth=2)
threshold_line, = ax.plot(x_vals, Y_vals, color="#f70202", linestyle="--", label="Umbral", linewidth=2)
ax.set_ylim(0, 0.4)
ax.set_xlim(0, max_frames)
ax.set_xlabel("Frames")
ax.set_ylabel("EAR")
ax.legend(loc="upper right")

def update_plot(ear, frame_num, ear_thr):
    """Actualizar la gráfica con nuevos valores de EAR"""
    ear_values.append(ear)
    frame_numbers.append(frame_num)
    if len(ear_values) > max_frames:
        ear_values.pop(0)
        frame_numbers.pop(0)

    EAR_curve.set_xdata(frame_numbers)
    EAR_curve.set_ydata(ear_values)
    threshold_line.set_xdata(frame_numbers)
    threshold_line.set_ydata([ear_thr if ear_thr else 0] * len(frame_numbers))

    ax.set_xlim(max(0, frame_num - max_frames), frame_num)

    canvas.draw()
    buf = canvas.buffer_rgba()
    plot_img = np.asarray(buf)
    plot_img = cv2.cvtColor(plot_img, cv2.COLOR_RGBA2BGR)
    return plot_img

# ==============================
# Inicialización de MediaPipe
# ==============================
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(refine_landmarks=True)

# Índices de los puntos clave de cada ojo
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

def get_eye_aspect_ratio(landmarks, eye_points):
    points = [landmarks[i] for i in eye_points]
    vertical_1 = np.linalg.norm(np.array([points[1].x, points[1].y]) -
                                np.array([points[5].x, points[5].y]))
    vertical_2 = np.linalg.norm(np.array([points[2].x, points[2].y]) -
                                np.array([points[4].x, points[4].y]))
    horizontal = np.linalg.norm(np.array([points[0].x, points[0].y]) -
                                np.array([points[3].x, points[3].y]))
    ear = (vertical_1 + vertical_2) / (2.0 * horizontal)
    return ear

# ==============================
# Inicializar captura de video
# ==============================
cap = cv2.VideoCapture(0)

blink_count = 0
frame_number = 0
blink_detected = False

# --- Variables para calibración ---
CALIBRATION_TIME = 4  # segundos de calibración
calib_start = time.time()
calib_ears = []
calibrated = False

# --- Variables para PERCLOS ---
WINDOW_SEC = 60
perclos_window = deque()
time_window = deque()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(frame_rgb)

    ear = None
    if results.multi_face_landmarks:
        for face_landmarks in results.multi_face_landmarks:
            left_ear = get_eye_aspect_ratio(face_landmarks.landmark, LEFT_EYE)
            right_ear = get_eye_aspect_ratio(face_landmarks.landmark, RIGHT_EYE)
            ear = (left_ear + right_ear) / 2.0

            # === DIBUJAR LANDMARKS DE LOS OJOS EN TODO EL VIDEO ===
            for idx in LEFT_EYE + RIGHT_EYE:
                lm = face_landmarks.landmark[idx]
                x = int(lm.x * frame.shape[1])
                y = int(lm.y * frame.shape[0])
                cv2.circle(frame, (x, y), 2, (0, 255, 0), -1)

            # ---- CALIBRACIÓN ----
            if not calibrated:
                calib_ears.append(ear)
                elapsed = time.time() - calib_start
                cv2.putText(frame, f"Calibrando: manten los ojos abiertos {int(CALIBRATION_TIME - elapsed)}s",
                            (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
                if elapsed >= CALIBRATION_TIME:
                    mu, sd = np.mean(calib_ears), np.std(calib_ears)
                    EAR_THRESHOLD = (mu - 1.25 * sd)*0.68
                    EAR_THRESHOLD = float(np.clip(EAR_THRESHOLD, 0.08, 0.45))
                    calibrated = True
                    print(f"Umbral calibrado EAR_THRESHOLD = {EAR_THRESHOLD:.3f}")

            else:
                # ---- DETECCIÓN DE PARPADEOS ----
                if ear < EAR_THRESHOLD:
                    if not blink_detected:
                        blink_count += 1
                        blink_detected = True
                        print(f"Parpadeo detectado, contador: {blink_count}")
                else:
                    blink_detected = False

                # ---- PERCLOS ----
                now = time.time()
                time_window.append(now)
                perclos_window.append(1 if ear < EAR_THRESHOLD else 0)
                while time_window and (now - time_window[0]) > WINDOW_SEC:
                    time_window.popleft()
                    perclos_window.popleft()
                perclos = sum(perclos_window) / max(1, len(perclos_window))

                # Mostrar métricas en pantalla
                cv2.putText(frame, f"Parpadeos: {blink_count}", (20, 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
                cv2.putText(frame, f"PERCLOS({WINDOW_SEC}s): {perclos*100:5.1f}%",
                            (20, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 255), 2)

    # ---- Gráfico EAR ----
    if ear is not None:
        plot_img = update_plot(ear, frame_number, EAR_THRESHOLD if calibrated else 0)
        plot_img_resized = cv2.resize(plot_img, (frame.shape[1], 200))
        combined = cv2.vconcat([frame, plot_img_resized])
    else:
        combined = frame

    cv2.imshow("Detección en Tiempo Real con EAR + Calibración + PERCLOS", combined)

    frame_number += 1
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


ModuleNotFoundError: No module named 'mediapipe'